In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q12/q12_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

### cell 2 (optimized for cudf) ###

# filter bounds
date1 = pd.Timestamp("1994-01-01")
date2 = pd.Timestamp("1995-01-01")

# build a single boolean mask using vectorized cudf operations
mask = (
    (lineitem.L_RECEIPTDATE >= date1)
    & (lineitem.L_RECEIPTDATE < date2)
    & (lineitem.L_COMMITDATE   < date2)
    & (lineitem.L_SHIPDATE      < date2)
    & (lineitem.L_SHIPDATE      < lineitem.L_COMMITDATE)
    & (lineitem.L_COMMITDATE    < lineitem.L_RECEIPTDATE)
    & lineitem.L_SHIPMODE.isin(["MAIL", "SHIP"])
)

# apply filter and join
flineitem = lineitem[mask]
jn = flineitem.merge(orders, left_on="L_ORDERKEY", right_on="O_ORDERKEY")

# build boolean flags for urgent vs non-urgent
urgent_mask = jn.O_ORDERPRIORITY.isin(["1-URGENT", "2-HIGH"])
jn["is_urgent"]     = urgent_mask
jn["is_not_urgent"] = ~urgent_mask

# group and aggregate both counts in one GPU pass
total = jn.groupby("L_SHIPMODE", as_index=False).agg(
    g1=("is_urgent",     "sum"),
    g2=("is_not_urgent", "sum")
)

print(total)